In [ ]:
from IPython.display import HTML
shell = get_ipython()

def adjust_font_size():
  display(HTML('''<style>
    body {
      font-size: 18px;
    }
  '''))

if adjust_font_size not in shell.events.callbacks['pre_execute']:
  shell.events.register('pre_execute', adjust_font_size)

In [ ]:
!pip install gradio
!pip install -q datasets transformers

Interface for Classifying Images into Its Classes

In [ ]:
import gradio as gr
from transformers import ViTForImageClassification, ViTImageProcessor

id2label = {0: 'airplane',
            1: 'automobile',
            2: 'bird',
            3: 'cat',
            4: 'deer',
            5: 'dog',
            6: 'frog',
            7: 'horse',
            8: 'ship',
            9: 'truck'}
label2id = {'airplane': 0,
            'automobile': 1,
            'bird': 2,
            'cat': 3,
            'deer': 4,
            'dog': 5,
            'frog': 6,
            'horse': 7,
            'ship': 8,
            'truck': 9}

In [ ]:
# using original model without fine-tuning
model_name = "google/vit-base-patch16-224-in21k"
model = ViTForImageClassification.from_pretrained(model_name,
                                                  id2label=id2label,
                                                  label2id=label2id)
vit_processor = ViTImageProcessor.from_pretrained(model_name)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
#using our fine-tuned model
from google.colab import drive
drive.mount('/content/drive')

model_name = "drive/MyDrive/model"
model = ViTForImageClassification.from_pretrained(model_name,
                                                  id2label=id2label,
                                                  label2id=label2id)
vit_processor = ViTImageProcessor.from_pretrained(model_name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'drive/MyDrive/model'. Use `repo_type` argument if needed.

In [ ]:
def image_mod(image):
    inputs = vit_processor(images=image, return_tensors="pt")
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class_idx = logits.argmax(-1).item()
    return model.config.id2label[predicted_class_idx]


demo = gr.Interface(
    image_mod,
    gr.Image(type="pil"),
    "text",
)

demo.launch(share=True)

In [ ]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

translation_pip = pipeline("translation", model=model,
                           tokenizer=tokenizer,
                           src_lang="en",
                           tgt_lang="fr",
                           max_length = 400)

def translate(text_to_translate):
    output = translation_pip(text_to_translate)
    return output[0]["translation_text"]

demo = gr.Interface(
    translate,
    "text",
    "text",
)

demo.launch(share=True)